In [ ]:
using NBInclude

In [ ]:
@nbinclude("s722.ipynb")

In [ ]:
@assert length(sol722) == 23

In [ ]:
# ============================================================
# Construct xval, yval, zval from sol722.
# Do NOT use undef here: an unexecuted/failing fill loop can leave
# invalid garbage rationals such as a//0, which makes QQ(a) fail.
# ============================================================

@assert length(sol722) == 23
@assert all(isassigned(sol722, k) for k in 1:23)

xval = zeros(Rational{Int}, 3, 3, 23)
yval = zeros(Rational{Int}, 3, 3, 23)
zval = zeros(Rational{Int}, 3, 3, 23)

for k in 1:23
    Uk, Vk, Wk = sol722[k]
    @assert size(Uk) == (3,3)
    @assert size(Vk) == (3,3)
    @assert size(Wk) == (3,3)

    for i in 1:3, j in 1:3
        xval[i,j,k] = Uk[i,j]
        yval[i,j,k] = Vk[i,j]
        zval[i,j,k] = Wk[i,j]
    end
end

println("xval, yval, zval constructed from sol722.")

In [ ]:
using Oscar

In [ ]:
R, x, y, z = polynomial_ring(
    QQ,
    :x => (1:3, 1:3, 1:23),
    :y => (1:3, 1:3, 1:23),
    :z => (1:3, 1:3, 1:23)
)

In [ ]:
#x[:,:,3] #test

In [ ]:
#kron(x[:,:,1],y[:,:,1],z[:,:,1])

In [ ]:
Svars = include("vars_Sgap2_oscar.jl")

In [ ]:
println("Number of variables in S = ", length(Svars))

In [ ]:
Sset = Set(Svars)

In [ ]:
function to_R(a::Rational{<:Integer})
    if denominator(a) == 0
        error("Cannot convert invalid Rational with denominator 0: $a. Rerun the xval/yval/zval construction cell.")
    end
    return R(QQ(a))
end

to_R(a::Integer) = R(QQ(a))

In [ ]:
PolyElem = typeof(x[1,1,1])

In [ ]:
U = fill(zero(R), 3, 3, 23)
V = fill(zero(R), 3, 3, 23)
W = fill(zero(R), 3, 3, 23)

In [ ]:

for k in 1:23, j in 1:3, i in 1:3
    U[i,j,k] = (x[i,j,k] in Sset) ? x[i,j,k] : to_R(xval[i,j,k])
    V[i,j,k] = (y[i,j,k] in Sset) ? y[i,j,k] : to_R(yval[i,j,k])
    W[i,j,k] = (z[i,j,k] in Sset) ? z[i,j,k] : to_R(zval[i,j,k])
end

println("Replacement finished.")

In [ ]:
B1 = kron(U[:,:,1], V[:,:,1], W[:,:,1])

In [ ]:
for k in 2:23
    B1 =B1+kron(U[:,:,k], V[:,:,k], W[:,:,k])
end
B1

In [ ]:
mm=3; nn=3; pp=3;

indices = [(i, j, k) for i in 1:mm for j in 1:nn for k in 1:pp]  #step2  for generating the polynomial system

tenset=[]

for (i, j, k) in indices
    matrix = zeros(Int, mm, nn)
    matrix[i,j]=1 
    matrix1 = matrix
    matrix = zeros(Int, nn, pp)
    matrix[j,k]=1 
    matrix2 = matrix
    matrix = zeros(Int, pp, mm)
    matrix[k,i]=1 
    matrix3 = matrix
   tensor_product = kron(kron(matrix1, matrix2), matrix3)
    push!(tenset, tensor_product)
end
T=sum(tenset)

In [ ]:
brent=vec(B1-T)

In [ ]:
# ============================================================
# Remove zero generators and duplicates up to nonzero QQ scalars
# ============================================================

function normalize_generator(f)
    iszero(f) && return f

    lc = leading_coefficient(f)

    # Make the leading coefficient equal to 1.
    # Thus f, -f, 2f, (1//4)f all have the same representative.
    return f * inv(lc)
end

In [ ]:
# Step 1: remove zero polynomials
Brent_nonzero = [ f for f in brent if !iszero(f) ]

In [ ]:
# Step 2: remove exact duplicates
Brent_exact_unique = unique(Brent_nonzero)

In [ ]:
# Step 3: normalize signs and all nonzero rational scalar multiples
Brent_normalized = [
    normalize_generator(f)
    for f in Brent_exact_unique
]

In [ ]:
# Step 4: remove duplicates after normalization
Brent_unique = unique(Brent_normalized)

In [ ]:
println("Original number:                 ", length(brent))
println("Number of nonzero polynomials:   ", length(Brent_nonzero))
println("After exact deduplication:       ", length(Brent_exact_unique))
println("After QQ-scalar deduplication:   ", length(Brent_unique))

In [ ]:
#f=Brent_unique[1]

In [ ]:
#vars(f)

In [ ]:
Slist = unique(Svars)

In [ ]:
neq = length(Brent_unique)

In [ ]:
nvar = length(Slist)

In [ ]:
var_to_idx = Dict(
    v => j
    for (j, v) in enumerate(Slist)
)

In [ ]:
g=graph(Undirected, neq + nvar)

In [ ]:
for (i, f) in enumerate(Brent_unique)
    for v in vars(f)

        if !haskey(var_to_idx, v)
            error(
                "Variable $v occurs in equation $i, " *
                "but it is not contained in Svars."
            )
        end

        variable_vertex = neq + var_to_idx[v]

       add_edge!(
            g,
            i,
            variable_vertex,
        )
    end
end

In [ ]:
independent_components=connected_components(g)

In [ ]:
println(
    "number of vertices = ",
    Oscar.n_vertices(g))

In [ ]:
println(
    "number of edges = ",
    Oscar.n_edges(g))

In [ ]:
println(
    "number of all connected components = ",
    length(independent_components)
)

In [ ]:
components = [C    
        for C in independent_components
        if any(vertex -> vertex <= neq, C)]

In [ ]:
println(
    "number of independent subsystems = ",
    length(components))

In [ ]:
blocks = [
    begin
        equation_indices = sort([
            vertex
            for vertex in C
            if vertex <= neq
        ])

        variable_indices = sort([
            vertex - neq
            for vertex in C
            if vertex > neq
        ])

        (
            equation_indices = equation_indices,
            variable_indices = variable_indices,
            equations = Brent_unique[equation_indices],
            variables = Slist[variable_indices],
        )
    end
    for C in components
]

In [ ]:
for (k, block) in enumerate(blocks)
    println(
        "Subsystem $k: ",
        length(block.equations),
        " equations, ",
        length(block.variables),
        " variables",
    )
end

In [ ]:
F1 = blocks[1].equations

In [ ]:
V1 = blocks[1].variables

In [ ]:
@show length(F1)

In [ ]:
@show length(V1)

In [ ]:
F1

In [ ]:
function save_oscar_subsystem(
    filename::AbstractString,
    F::AbstractVector,
    V::AbstractVector;
    equation_name::Symbol = :Brent_unique_subsystem,
    variable_name::Symbol = :Svars,
)
    isempty(F) && error("The polynomial vector F is empty.")
    isempty(V) && error("The variable vector V is empty.")

    eq_name = string(equation_name)
    var_name = string(variable_name)

    open(filename, "w") do io
        println(io, "# Generated from .")
        println(io, "# Run this file after creating the same Oscar ring R and variables x, y, z.")
        println(io)

        # Save the variables of subsystem.
        println(io, var_name, " = [")
        for v in V
            println(io, "    ", string(v), ",")
        end
        println(io, "]")
        println(io)

        # Save the equations of subsystem.
        println(io, eq_name, " = [")
        for f in F
            println(io, "    ", string(f), ",")
        end
        println(io, "]")
        println(io)

        # Make include(filename) return the equation vector.
        println(io, eq_name)
    end

    return abspath(filename)
end

In [ ]:
output_file = save_oscar_subsystem("Brent_unique_subsystem_1_gap2.jl", F1, V1)

In [ ]:
println("Saved ", length(F1), " equations and ", length(V1),  " variables to: ", output_file)

In [ ]:
F2 = blocks[2].equations

V2 = blocks[2].variables

output_file = save_oscar_subsystem("Brent_unique_subsystem_2_gap2.jl", F2, V2)

# F3 = blocks[3].equations

# V3 = blocks[3].variables

# output_file = save_oscar_subsystem("Brent_unique_subsystem_2_gap10.jl", F2, V2)

# output_file = save_oscar_subsystem("Brent_unique_subsystem_3_gap10.jl", F3, V3)